# Prompt Evaluation Dataset Generation with Gemini

This notebook demonstrates how to use Gemini to generate an evaluation dataset for various coding and configuration tasks.

### 1. Setup and Initialization
We start by loading environmental variables and initializing the Google GenAI client. We specify `gemini-2.5-flash` as our default model for its speed and efficiency.

In [1]:
import os
import json
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import display, Markdown

# Load API Key from .env
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Initialize the Client
client = genai.Client(api_key=api_key)
model_id = "gemini-2.5-flash"

### 2. Chat Helper Function
This helper function wraps the `generate_content` call to simplify interactions with the model. It handles configuration parameters like `temperature` and `system_instruction`.

In [2]:
def chat(prompt, system_instruction=None, temperature=1.0):
    """
    Simple wrapper for generating content with Gemini.
    """
    config = types.GenerateContentConfig(
        temperature=temperature,
        system_instruction=system_instruction
    )
    
    response = client.models.generate_content(
        model=model_id,
        contents=prompt,
        config=config
    )
    return response.text

### 3. Dataset Generation Logic
This function prompts Gemini to create a specific evaluation dataset for AWS-related tasks. It includes logic to clean the output (removing markdown blocks) and saves the resulting JSON to the `data/` directory.

In [7]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing a task that requires Python, JSON, or a Regex to complete.

Example output structure:
[
    {
        "task": "Description of task",
        "type": "python|json|regex"
    },
    ...
]

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks related to AWS services (S3, Lambda, EC2, etc.).
* Please generate 5 unique tasks.
"""
    
    print("Generating dataset...")
    response_text = chat(
        prompt=prompt, 
        system_instruction="You are a helpful assistant that outputs ONLY valid raw JSON.",
        temperature=0.7
    )
    
    # Clean response in case model includes markdown formatting
    clean_json = response_text.replace("```json", "").replace("```", "").strip()
    
    try:
        dataset = json.loads(clean_json)
        
        # Ensure data directory exists
        os.makedirs("../../data", exist_ok=True)
        
        file_path = "../../data/eval_dataset.json"
        with open(file_path, "w") as f:
            json.dump(dataset, f, indent=4)
            
        print(f"Dataset successfully saved to {file_path}")
        return dataset
    except Exception as e:
        print(f"Error parsing or saving JSON: {e}")
        print("Raw Response:", response_text)
        return None

### 4. Execution
Finally, we run the generation function and display the results directly in the notebook.

In [8]:
# Execute the generation
dataset = generate_dataset()
if dataset:
    display(dataset)

Generating dataset...
Dataset successfully saved to ../../data/eval_dataset.json


[{'task': "Write a Python function using boto3 to retrieve the current state (e.g., 'running', 'stopped') of a given EC2 instance ID.",
  'type': 'python'},
 {'task': 'Define an IAM policy document that grants read-only access to all objects within a specific S3 bucket, allowing `s3:GetObject` and `s3:ListBucket` actions.',
  'type': 'json'},
 {'task': "Extract the `Request ID` (e.g., `5443F93406C37BA8`) from an S3 error message such as 'Access Denied (Request ID: 5443F93406C37BA8)'.",
  'type': 'regex'},
 {'task': 'Write a Python function using boto3 to publish a message to an SNS topic, given the topic ARN and the message body string.',
  'type': 'python'},
 {'task': 'Generate a CloudFormation resource definition for an AWS Lambda function, specifying its `Runtime`, `Handler`, and `Code` S3 bucket/key properties.',
  'type': 'json'}]